# Home Credit: Model tuning

**Completed: eight candidates × five temporal folds; five control folds reused.**

Does bounded tuning improve stability without removing validated feature blocks? All 700 features were retained. This is an executed review of aggregate results, not a retraining notebook. The untouched final holdout is weeks 73–91.

In [ ]:
import hashlib
import json
from pathlib import Path

import pandas as pd
from IPython.display import SVG, display

root = next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / "configs/model_tuning.json").is_file()
)
evidence = root / "reports/model_tuning/metrics.json"
assert (
    hashlib.sha256(evidence.read_bytes()).hexdigest()
    == "5b2dbc4b5dfcbdcfc19e5d609cf6742065b4642f969ef53df83b2d726acb8295"
)
result = json.loads(evidence.read_text())
assert result["complete"] and not result["smoke"]
assert result["outer_holdout_touched"] is False
assert result["completed_new_model_folds"] == 40
print(result["scope"])
table = pd.DataFrame(result["rows"]).sort_values(
    "mean_fold_stability", ascending=False, kind="stable"
)
display(table.round(6).reset_index(drop=True))

## Improvement and its limits

The primary objective is the mean of five official stability scores: mean weekly Gini + 88 × min(temporal slope, 0) − 0.5 × residual standard deviation. Pooled OOF ROC AUC and average precision support ranking evaluation; Brier and log loss support probability evaluation.

In [ ]:
import matplotlib

matplotlib.use("Agg")
from io import StringIO

import matplotlib.pyplot as plt

plt.rcParams["svg.fonttype"] = "none"
plt.rcParams["svg.hashsalt"] = "home-credit-tuning-review"
fig, ax = plt.subplots(figsize=(10, 4.5), layout="constrained")
ranked = table.iloc[::-1]
control = float(table.loc[table["candidate"] == "control", "mean_fold_stability"].iloc[0])
ax.barh(ranked["candidate"], ranked["mean_fold_stability"] - control)
ax.axvline(0, linewidth=1)
ax.set(
    xlabel="Change in mean official fold stability versus control",
    title="Development tuning | which changes improved stability?",
)
ax.spines[["top", "right"]].set_visible(False)
buffer = StringIO()
fig.savefig(buffer, format="svg", metadata={"Date": None})
display(SVG(data=buffer.getvalue()))
plt.close(fig)

In [ ]:
folds = pd.DataFrame(result["folds"])
display(folds.round(6))
fig, ax = plt.subplots(figsize=(10, 4.5), layout="constrained")
for name in ["control", "trial_006"]:
    ax.plot(folds["fold"], folds[name], "o-", label=name)
ax.set(
    xlabel="Expanding temporal fold",
    ylabel="Official stability score",
    title="Where did the selected configuration improve?",
)
ax.set_xticks(folds["fold"])
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
buffer = StringIO()
fig.savefig(buffer, format="svg", metadata={"Date": None})
display(SVG(data=buffer.getvalue()))
plt.close(fig)
assert int((folds["delta"] > 0).sum()) == 3
print(result["interpretation"])

## Decision

Retain `trial_006` as the tuned LightGBM development incumbent. Mean fold stability rises from **0.585188 to 0.601238**, but folds 3 and 4 regress slightly. Trial 004 has marginally better Brier and log loss; the official stability objective, not probability error, selects trial 006.

Next, compare a small, fixed set of blends using saved LightGBM, XGBoost and CatBoost predictions. Do not repeat the completed 40 fits. The next launcher is `bash scripts/start_model_selection.sh --bucket YOUR_ARTIFACT_BUCKET`. Successful completion publishes the canonical `notebooks/08_model_selection.ipynb` and a durable S3-backed run report.

After development choices are frozen: evaluate the holdout once, refit and validate the complete inference pipeline, then let the owner generate, validate and download a submission from notebook code. **No Kaggle upload is automated.** A new leaderboard score is not implied by these development results.

In [ ]:
print("Selected parameters:")
print(json.dumps(result["selected_parameters"], indent=2, sort_keys=True))
print("Training commit:", result["training_git_commit"])
print("Full source study SHA-256:", result["source_study_sha256"])
print("Study completed UTC:", result["completed_utc"])
print("The committed JSON is an aggregate excerpt, not the full S3 ledger.")